# Model Interpretation: Deepfake Image Classification:

### Import Libraries:

In [14]:
import torch
import torch.nn as nn
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
import pandas as pd
import joblib

from pathlib import Path
from PIL import Image
from torchvision import transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


### Load the dataset:

In [15]:
# --------------------------------------------------
# Change only these values
# --------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final")

IMG_DIR = PROJECT_DIR / "data" / "raw_images"
TEST_CSV = PROJECT_DIR / "data" / "test.csv"
FINAL_CSV = PROJECT_DIR / "data" / "final_dataset.csv"
MODEL_PATH = PROJECT_DIR / "models" / "resnet50_model.pth"

NUM_REAL = 2           # how many random REAL images to show
NUM_FAKE = 2           # how many random FAKE images to show

USE_TEST_SET = True    # True  -> pick from the 834 test images (the model has never trained on them)
                       # False -> pick from all 5,557 images in the folder

THRESHOLD = 0.5        # probability of FAKE >= THRESHOLD -> FAKE, otherwise REAL

SEED = None            # None -> different images every run
                       # a number (for example 42) -> the same images every run

### Load the ResNet50 Model:

In [16]:
# --------------------------------------------------
# 1. Device
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# --------------------------------------------------
# 2. Image preprocessing (same as validation / test)
# --------------------------------------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# --------------------------------------------------
# 3. Load trained ResNet50
# --------------------------------------------------

resnet_model = models.resnet50(weights=None)

resnet_model.fc = nn.Sequential( #type: ignore
    nn.Linear(resnet_model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

resnet_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

resnet_model = resnet_model.to(device)
resnet_model.eval()

print("ResNet50 model loaded successfully!")


# --------------------------------------------------
# 4. Grad-CAM on the last convolutional block
# --------------------------------------------------

cam = GradCAM(
    model=resnet_model,
    target_layers=[resnet_model.layer4[-1]]
)

print("Grad-CAM ready on layer4[-1]")

Using device: cuda
ResNet50 model loaded successfully!
Grad-CAM ready on layer4[-1]


### Grad-CAM for one image:

The model has **one output**: the score for FAKE. Grad-CAM needs to be told *which* score to explain:

- If the model predicts **FAKE**, we explain the FAKE score (regions that pushed it towards FAKE).
- If the model predicts **REAL**, we explain the REAL score (regions that pushed it towards REAL).

If we always explained the FAKE score, REAL images would show an almost empty (all blue) heatmap, because a REAL image has very little evidence for FAKE.

In [17]:
# --------------------------------------------------
# Which score should Grad-CAM explain?
# --------------------------------------------------

class BinaryTarget:

    def __init__(self, predicted_fake):
        self.sign = 1.0 if predicted_fake else -1.0

    def __call__(self, model_output):
        return self.sign * model_output[0]


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])


# --------------------------------------------------
# Predict one image and build its Grad-CAM
# --------------------------------------------------

def explain_image(image_path, true_label):

    # Open and preprocess
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)  # type: ignore

    # Prediction
    with torch.no_grad():
        prob_fake = torch.sigmoid(resnet_model(input_tensor)).item()

    prediction = "FAKE" if prob_fake >= THRESHOLD else "REAL"

    # Grad-CAM for the PREDICTED class
    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[BinaryTarget(prediction == "FAKE")]  # type: ignore
    )[0]

    # The image exactly as the model saw it (undo the normalization)
    rgb_image = input_tensor[0].cpu().permute(1, 2, 0).numpy()
    rgb_image = np.clip(
        rgb_image * IMAGENET_STD + IMAGENET_MEAN, 0, 1
    ).astype(np.float32)

    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )

    return {
        "true_label": true_label,
        "prediction": prediction,
        "prob_fake": prob_fake,
        "original": (rgb_image * 255).astype(np.uint8),
        "gradcam": visualization
    }

### Show random images (2 REAL and 2 FAKE):

In [18]:
# --------------------------------------------------
# 1. Pick random images (new ones every time)
# --------------------------------------------------

csv_path = TEST_CSV if USE_TEST_SET else FINAL_CSV

data_df = pd.read_csv(csv_path)

real_df = data_df[data_df["label"] == "REAL"]
fake_df = data_df[data_df["label"] == "FAKE"]

sample_df = pd.concat([
    real_df.sample(n=min(NUM_REAL, len(real_df)), random_state=SEED),
    fake_df.sample(n=min(NUM_FAKE, len(fake_df)), random_state=SEED)
])

print("Picked from:", csv_path.name, f"({len(data_df)} images)\n")


# --------------------------------------------------
# 2. Predict + Grad-CAM
# --------------------------------------------------

results = []

for row in sample_df.itertuples():

    image_path = IMG_DIR / f"{row.image_id}.jpg"

    result = explain_image(image_path, row.label)
    result["image_id"] = row.image_id
    results.append(result)

    status = "correct" if result["prediction"] == result["true_label"] else "WRONG"

    print(
        f"{image_path.name} -> "
        f"True: {result['true_label']} | "
        f"Predicted: {result['prediction']} ({status}) | "
        f"FAKE Probability: {result['prob_fake'] * 100:.2f}%"
    )


# --------------------------------------------------
# 3. Plotly figure: top row = REAL images, bottom row = FAKE images
# --------------------------------------------------

row_groups = [
    [r for r in results if r["true_label"] == "REAL"],
    [r for r in results if r["true_label"] == "FAKE"]
]

per_row = max(NUM_REAL, NUM_FAKE)      # images per row (each image takes 2 columns)

subplot_titles = []

for group in row_groups:

    for k in range(per_row):

        if k < len(group):

            r = group[k]
            status = "correct" if r["prediction"] == r["true_label"] else "WRONG"

            subplot_titles.append(
                f"True: {r['true_label']} | Pred: {r['prediction']}"
                f"<br>{status} | P(FAKE) = {r['prob_fake']:.3f}"
            )
            subplot_titles.append(f"Grad-CAM for {r['prediction']}")

        else:
            subplot_titles += ["", ""]

fig = make_subplots(
    rows=2,
    cols=2 * per_row,
    subplot_titles=subplot_titles,
    vertical_spacing=0.12,
    horizontal_spacing=0.015
)

for row_number, group in enumerate(row_groups, start=1):

    for k, r in enumerate(group):

        fig.add_trace(go.Image(z=r["original"]), row=row_number, col=2 * k + 1)
        fig.add_trace(go.Image(z=r["gradcam"]), row=row_number, col=2 * k + 2)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_annotations(font_size=12)

fig.update_layout(
    title="ResNet50 - Grad-CAM on random images (top row: REAL, bottom row: FAKE)",
    width=280 * 2 * per_row + 40,
    height=760
)

fig.show()

joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\models\grad_cam_fig.pkl")
print(" grad_cam_fig saved as pkl file")


Picked from: test.csv (834 images)

1284.jpg -> True: REAL | Predicted: REAL (correct) | FAKE Probability: 0.03%
2175.jpg -> True: REAL | Predicted: REAL (correct) | FAKE Probability: 0.11%
4587.jpg -> True: FAKE | Predicted: FAKE (correct) | FAKE Probability: 100.00%
4714.jpg -> True: FAKE | Predicted: FAKE (correct) | FAKE Probability: 100.00%


 grad_cam_fig saved as pkl file
